<a href="https://colab.research.google.com/github/vedaaanggshetty/DocGenerator/blob/main/EmoChat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Wed Jan  7 04:56:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece fastapi gradio huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.3 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
login()

In [4]:
%%writefile model.py
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = "EmoCareAI/ChatPsychiatrist"

# ---- Quantization config (correct) ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# ---- Load tokenizer & model ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

def generate_response(user_message: str) -> str:
    prompt = f"""
You are ChatPsychiatrist, an empathetic AI mental health support assistant.

Guidelines:
- Be calm, supportive, and non-judgmental
- Focus on emotions, thoughts, and coping strategies
- Do NOT discuss unrelated topics
- Do NOT give medical diagnoses
- Ask gentle follow-up questions when appropriate

User: {user_message}
Assistant:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("Assistant:")[-1].strip()

Writing model.py


In [5]:
pip install fastapi uvicorn pyngrok

In [6]:
from fastapi import FastAPI
from pydantic import BaseModel
from model import generate_response

app = FastAPI()

class ChatRequest(BaseModel):
    message: str
    history: list[str] = []

@app.post("/chat")
def chat(req: ChatRequest):
    reply = generate_response(req.message)
    return {"reply": reply}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggin

...

pytorch_model-00001-of-00003.bin:   0%|          | 0.00/9.88G [00:00<?, ?B/s]

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/9.89G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [7]:
import threading
import uvicorn

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

In [8]:
import requests

requests.post(
    "http://localhost:8000/chat",
    json={"message": "hi", "history": []}
).json()

INFO:     127.0.0.1:60844 - "POST /chat HTTP/1.1" 200 OK


{'reply': "Hello! It's nice to meet you. I'm here to help you with any emotional or mental health concerns you may have. Please feel free to share what's been going on in your life and how you've been feeling lately."}

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("37t7RB2zceZBnczRff01MgvS9Qo_691Np1UX5Bfyi5E7zqj27")

In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://fred-athetoid-setsuko.ngrok-free.dev" -> "http://localhost:8000"
